# 예제 01. 문장을 숫자로 바꾸기
빅데이터프로그래밍 · 12주차

## 목표
- 문장을 토큰으로 자른다
- 단어 사전을 만든다
- 문장을 번호 목록으로 바꾼다

모델은 문장을 직접 읽지 못합니다. `문장 → 토큰 → 번호 → Tensor` 순서로 바꿉니다.


In [ ]:
import torch
import re
from collections import Counter

torch.manual_seed(42)


## 1. 토큰화 — 문장을 단어로 자릅니다


In [ ]:
text = "이 영화는 정말 재미있었다. 배우들의 연기가 훌륭했다!"

print("공백으로 자르기:", text.split())


In [ ]:
def tokenize(s):
    s = s.lower()
    s = re.sub(r"[^가-힣a-z0-9\s]", " ", s)     # 기호 제거
    return s.split()

print(tokenize(text))


기호를 남기면 `재미있었다.` 와 `재미있었다` 가 서로 다른 단어가 됩니다.


In [ ]:
sentences = [
    "이 영화는 정말 재미있었다",
    "배우 연기가 훌륭했다",
    "너무 지루하고 실망스러웠다",
    "재미없고 시간 낭비였다",
    "정말 좋은 영화였다",
]
tokens = [tokenize(s) for s in sentences]
for t in tokens:
    print(t)


## 2. 단어 빈도 세기


In [ ]:
counter = Counter(w for t in tokens for w in t)
print("전체 단어 수 :", sum(counter.values()))
print("서로 다른 단어:", len(counter))
print("\n자주 나온 순:")
for word, n in counter.most_common(5):
    print(f"  {word:8s} {n}회")


## 3. 단어 사전 만들기
번호 0과 1은 특별한 토큰에 예약합니다.

| 토큰 | 번호 | 하는 일 |
| --- | --- | --- |
| `<PAD>` | 0 | 짧은 문장을 채우는 빈칸 |
| `<UNK>` | 1 | 사전에 없는 단어 |


In [ ]:
PAD, UNK = 0, 1

def build_vocab(token_lists, min_freq=1):
    counter = Counter(w for t in token_lists for w in t)
    vocab = {"<PAD>": PAD, "<UNK>": UNK}
    for word, n in counter.most_common():
        if n >= min_freq:
            vocab[word] = len(vocab)
    return vocab


vocab = build_vocab(tokens)
print("사전 크기:", len(vocab))
for word, idx in list(vocab.items())[:10]:
    print(f"  {idx:2d}  {word}")


## 4. 문장을 번호로


In [ ]:
def encode(tokens, vocab):
    return [vocab.get(w, UNK) for w in tokens]

for s, t in zip(sentences, tokens):
    print(f"{s}\n  → {encode(t, vocab)}\n")


## 5. 사전에 없는 단어는 UNK


In [ ]:
new = tokenize("이 영화는 최고의 걸작이었다")
print("토큰:", new)
print("번호:", encode(new, vocab))
print("\n'최고의' 와 '걸작이었다' 는 사전에 없어 1(UNK)이 됩니다")


## 6. 되돌려 보기 — 사전이 제대로 만들어졌는지 확인


In [ ]:
itos = {i: w for w, i in vocab.items()}

ids = encode(tokens[0], vocab)
print("번호:", ids)
print("복원:", [itos[i] for i in ids])


## 7. min_freq — 드문 단어를 잘라내기
한 번만 나온 단어를 사전에 넣으면 사전이 커지고 학습이 어려워집니다.


In [ ]:
import pandas as pd

big_tokens = [tokenize(s) for s in sentences * 3 + ["희귀한 단어 하나"]]
rows = []
for mf in [1, 2, 3]:
    v = build_vocab(big_tokens, min_freq=mf)
    rows.append({"min_freq": mf, "사전 크기": len(v)})
pd.DataFrame(rows)


## 직접 해보기
1. 자기 문장 5개로 사전을 만들고 번호로 바꿔 보세요.
2. `tokenize` 를 고쳐 한 글자 단어를 버리도록 만들어 보세요.
3. 사전에 없는 단어가 몇 개인지 세는 코드를 작성하세요.


In [ ]:
# 여기에 작성하세요
